# Jour 5 — Ajout d’un outil externe (search / docs)
### Objectif
Connecter un outil de recherche externe (mock ou API HTTP) à un agent OpenAI.

## 1️⃣ Présentation du concept
Les **tools** permettent aux agents d’étendre leurs capacités en exécutant des actions externes.
Exemples :
- Requêtes HTTP (ex : API de recherche, météo, base documentaire)
- Exécution de code local
- Lecture d’une base de connaissances (RAG, vector store)

Dans ce notebook, nous allons :
1. Créer un mock d’outil `search_tool`
2. L’intégrer à un agent (mode Python)
3. Puis en mode LangChain pour comparer.

In [1]:

# 🧩 Mode 1 : Agent avec tool mock (OpenAI direct)

from openai import OpenAI
import json

client = OpenAI()

# Définition du tool "search_tool"
def search_tool(query: str):
    mock_results = {
        "agents": ["AgentGPT", "AutoGPT", "LangChain Agent"],
        "mcp": ["Model Context Protocol", "Claude MCP", "OpenAI Tools"]
    }
    return mock_results.get(query.lower(), ["Aucun résultat trouvé"])

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_tool",
            "description": "Recherche dans un mock de base documentaire.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Sujet à rechercher"}
                },
                "required": ["query"]
            }
        }
    }
]

# Appel de l'agent OpenAI avec un tool
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "Tu es un agent de recherche documentaire."},
        {"role": "user", "content": "Cherche des informations sur les agents"}
    ],
    tools=tools
)

print(response.choices[0].message)


ModuleNotFoundError: No module named 'openai'

## 2️⃣ Mode 2 : Version LangChain
Nous allons implémenter le même comportement en utilisant LangChain Tools.

In [ ]:

# 🧩 Mode 2 : LangChain Tool

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType

@tool("search_tool", return_direct=False)
def search_tool(query: str) -> str:
    """Recherche mockée dans une base documentaire."""
    mock_results = {
        "agents": ["AgentGPT", "AutoGPT", "LangChain Agent"],
        "mcp": ["Model Context Protocol", "Claude MCP", "OpenAI Tools"]
    }
    return str(mock_results.get(query.lower(), ["Aucun résultat trouvé"]))

llm = ChatOpenAI(model="gpt-4o-mini")
agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent.run("Cherche des informations sur le MCP")


## ✅ Résumé
- Création d’un tool mock `search_tool`
- Intégration dans un agent OpenAI (direct)
- Intégration dans un agent LangChain

> Prochain jour : Connexion à un outil HTTP réel (ex: DuckDuckGo Search API).